In [1]:
import json
from typing import Dict, Any, Set, List, Tuple

In [2]:
def get_schema_keys(data: Any, path: str = "") -> Set[str]:
    """
    JSON 데이터에서 모든 키의 경로를 추출합니다.
    
    Args:
        data: JSON 데이터
        path: 현재 경로
    
    Returns:
        모든 키 경로의 집합
    """
    keys = set()
    
    if isinstance(data, dict):
        for key, value in data.items():
            current_path = f"{path}.{key}" if path else key
            keys.add(current_path)
            
            # 재귀적으로 하위 구조 탐색
            if isinstance(value, (dict, list)):
                keys.update(get_schema_keys(value, current_path))
    
    elif isinstance(data, list):
        # 리스트의 경우 첫 번째 요소의 구조를 기준으로 함
        if data:
            first_item = data[0]
            if isinstance(first_item, (dict, list)):
                keys.update(get_schema_keys(first_item, f"{path}[0]"))
    
    return keys

def compare_json_schemas(json1: Dict[Any, Any], json2: Dict[Any, Any]) -> Dict[str, Any]:
    """
    두 JSON의 스키마를 비교합니다.
    
    Args:
        json1: 첫 번째 JSON 데이터
        json2: 두 번째 JSON 데이터
    
    Returns:
        비교 결과를 담은 딕셔너리
    """
    schema1_keys = get_schema_keys(json1)
    schema2_keys = get_schema_keys(json2)
    
    # 추가된 키들
    added_keys = schema2_keys - schema1_keys
    
    # 삭제된 키들
    removed_keys = schema1_keys - schema2_keys
    
    # 공통 키들
    common_keys = schema1_keys & schema2_keys
    
    # 변경 여부
    has_changes = len(added_keys) > 0 or len(removed_keys) > 0
    
    return {
        "has_schema_changes": has_changes,
        "added_keys": sorted(list(added_keys)),
        "removed_keys": sorted(list(removed_keys)),
        "common_keys": sorted(list(common_keys)),
        "total_keys_json1": len(schema1_keys),
        "total_keys_json2": len(schema2_keys)
    }

def compare_json_files(file1_path: str, file2_path: str) -> Dict[str, Any]:
    """
    두 JSON 파일의 스키마를 비교합니다.
    
    Args:
        file1_path: 첫 번째 JSON 파일 경로
        file2_path: 두 번째 JSON 파일 경로
    
    Returns:
        비교 결과
    """
    try:
        with open(file1_path, 'r', encoding='utf-8') as f1:
            json1 = json.load(f1)
        
        with open(file2_path, 'r', encoding='utf-8') as f2:
            json2 = json.load(f2)
        
        return compare_json_schemas(json1, json2)
    
    except FileNotFoundError as e:
        return {"error": f"파일을 찾을 수 없습니다: {e}"}
    except json.JSONDecodeError as e:
        return {"error": f"JSON 파싱 오류: {e}"}
    except Exception as e:
        return {"error": f"예상치 못한 오류: {e}"}

def print_comparison_result(result: Dict[str, Any]) -> None:
    """비교 결과를 보기 좋게 출력합니다."""
    if "error" in result:
        print(f"❌ 오류: {result['error']}")
        return
    
    print("=" * 50)
    print("JSON 스키마 비교 결과")
    print("=" * 50)
    
    if result["has_schema_changes"]:
        print("🔄 스키마 변경이 감지되었습니다!")
    else:
        print("✅ 스키마에 변경이 없습니다.")
    
    print(f"\n📊 통계:")
    print(f"  - JSON1 총 키 개수: {result['total_keys_json1']}")
    print(f"  - JSON2 총 키 개수: {result['total_keys_json2']}")
    print(f"  - 공통 키 개수: {len(result['common_keys'])}")
    
    if result["added_keys"]:
        print(f"\n➕ 추가된 키 ({len(result['added_keys'])}개):")
        for key in result["added_keys"]:
            print(f"  + {key}")
    
    if result["removed_keys"]:
        print(f"\n➖ 삭제된 키 ({len(result['removed_keys'])}개):")
        for key in result["removed_keys"]:
            print(f"  - {key}")
    
    if not result["added_keys"] and not result["removed_keys"]:
        print(f"\n✨ 모든 키가 동일합니다.")


In [6]:
# 사용 예시
# 파일 경로 지정
file1 = "notebooks/dataload/prod_info_0219.json"
file2 = "notebooks/dataload/mobile_plan_info_20250705.json"

# 비교 실행
result = compare_json_files(file1, file2)

# 결과 출력
print_comparison_result(result)

result = compare_json_files(file2, file1)

# 결과 출력
print_comparison_result(result)


JSON 스키마 비교 결과
🔄 스키마 변경이 감지되었습니다!

📊 통계:
  - JSON1 총 키 개수: 358
  - JSON2 총 키 개수: 370
  - 공통 키 개수: 352

➕ 추가된 키 (18개):
  + [0].autoProductChange[0].changeRule
  + [0].autoProductChange[0].changeRule.date
  + [0].autoProductChange[0].changeRule.dateBase
  + [0].autoProductChange[0].changeRule.status
  + [0].autoProductChange[0].productNameAfterChange
  + [0].billingInfo.billingItem.valueList
  + [0].billingInfo.billingItemOnInvoice.valueList
  + [0].customerInfo.onboardingCustomer.welfareBenefitRule
  + [0].optionData.dataOptionProvidingMethod[0].pmDataOptionCode
  + [0].processInfo.devicePurchase
  + [0].processInfo.devicePurchase.devicePurchaseYn
  + [0].processInfo.subscriptionModificationType
  + [0].processInfo.subscriptionModificationType[0].eligibility
  + [0].processInfo.subscriptionModificationType[0].subscriptionCondition
  + [0].processInfo.subscriptionModificationType[0].subscriptionDateCriteria
  + [0].processInfo.subscriptionModificationType[0].subscriptionMethod
  + [0].pr